In [68]:
import math, itertools, random
import numpy as np
import pandas as pd

# Try imports for solvers
has_ortools = False
has_pulp = False
try:
    from ortools.linear_solver import pywraplp
    has_ortools = True
except Exception as e:
    try:
        import pulp
        has_pulp = True
    except Exception as e2:
        pass

import torch
import torch.nn as nn
import torch.nn.functional as F

In [69]:
class SimpleDQN(nn.Module):
    def __init__(self, input_dim, hidden=[64,32]):
        super().__init__()
        layers = []
        cur = input_dim
        for h in hidden:
            layers.append(nn.Linear(cur,h))
            layers.append(nn.ReLU())
            cur = h
        layers.append(nn.Linear(cur,1))  # scalar Q for a (s,a) pair
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)  # (...,) shape

In [70]:
# ----------------------
# Fake environment & features for demonstration
# ----------------------
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

# Suppose we have 8 candidate tiles/items to possibly cache
n_items = 8

# For each item, define:
sizes = np.random.randint(1,6, size=n_items)           # storage size units
latency_if_not_cached = np.random.uniform(50,500,size=n_items)  # ms cost if not cached
benefit = latency_if_not_cached  # simplifying: benefit of caching equals saved latency
capacity = 15   # total storage units available at the cache

# Feature vector for each (s,a) pair: we'll use [size, recent_requests, popularity_est, benefit]
recent_requests = np.random.randint(0,20,size=n_items)
popularity = np.random.rand(n_items)

features = np.stack([sizes, recent_requests, popularity*100, benefit], axis=1).astype(np.float32)
# Normalize features (simple)
features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-6)


In [71]:
# Create a DQN and compute Q-values for each candidate
dqn = SimpleDQN(input_dim=features.shape[1])
# random init is fine for demo; in practice load trained weights
with torch.no_grad():
    x = torch.tensor(features)
    q_values = dqn(x).numpy()

# Choose threshold phi to create DQN mask (tunable)
phi = np.percentile(q_values, 60)   # keep top 40% suggested items as mask
mask = (q_values >= phi).astype(int)

working_set = np.arange(n_items)



In [72]:

force_one = np.zeros(n_items, dtype=int)
force_zero = np.zeros(n_items, dtype=int)
for i in working_set:
    if mask[i]==1:
        force_one[i]=1
    else:
        force_zero[i]=1


In [73]:
def solve_pruned_ilp_ortools(sizes, benefit, capacity, force_one, force_zero):
    solver = pywraplp.Solver.CreateSolver('SCIP') or pywraplp.Solver.CreateSolver('CBC_MIXED_INTEGER_PROGRAMMING')
    if solver is None:
        raise RuntimeError("OR-Tools solver not available")
    n = len(sizes)
    L = [solver.IntVar(0,1,f"L_{i}") for i in range(n)]
    # capacity
    solver.Add(sum(sizes[i]*L[i] for i in range(n)) <= capacity)
    # force constraints
    for i in range(n):
        if force_one[i]:
            solver.Add(L[i] == 1)
        if force_zero[i]:
            solver.Add(L[i] == 0)
    # maximize benefit (equiv to minimize latency)
    objective = solver.Objective()
    for i in range(n):
        objective.SetCoefficient(L[i], benefit[i])
    objective.SetMaximization()
    status = solver.Solve()
    if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
        sol = np.array([int(L[i].solution_value()) for i in range(n)])
        return sol, solver.Objective().Value()
    else:
        raise RuntimeError("No solution found by OR-Tools")

In [74]:
def solve_pruned_ilp_pulp(sizes, benefit, capacity, force_one, force_zero):
    import pulp
    prob = pulp.LpProblem("pruned_cache", pulp.LpMaximize)
    n = len(sizes)
    L = [pulp.LpVariable(f"L_{i}", cat='Binary') for i in range(n)]
    prob += pulp.lpSum([benefit[i]*L[i] for i in range(n)])
    prob += pulp.lpSum([sizes[i]*L[i] for i in range(n)]) <= capacity
    for i in range(n):
        if force_one[i]:
            prob += L[i] == 1
        if force_zero[i]:
            prob += L[i] == 0
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    sol = np.array([int(pulp.value(L[i])) for i in range(n)])
    obj = pulp.value(prob.objective)
    return sol, obj

In [75]:
def solve_pruned_ilp_bruteforce(sizes, benefit, capacity, force_one, force_zero):
    n = len(sizes)
    best_obj = -1e9
    best_sol = None
    # brute force over subsets (only ok for n<=20)
    for bits in range(1<<n):
        sel = np.array([1 if (bits>>i)&1 else 0 for i in range(n)], dtype=int)

        # check forced constraints
        if np.any((force_one==1) & (sel==0)): continue
        if np.any((force_zero==1) & (sel==1)): continue
        if sel.dot(sizes) > capacity: continue

        obj = sel.dot(benefit)
        if obj > best_obj:
            best_obj = obj
            best_sol = sel.copy()
    if best_sol is None:
        raise RuntimeError("No feasible solution found by brute force")
    return best_sol, best_obj

force_one = np.zeros(n_items, dtype=int)
force_zero = np.zeros(n_items, dtype=int)
for i in working_set:
    if mask[i]==1:
        force_one[i]=1
    else:
        force_zero[i]=1

In [ ]:
sol, obj = solve_pruned_ilp_pulp(sizes, benefit, capacity, force_one, force_zero)
solver_used = "OR-Tools"

# sol, obj = solve_pruned_ilp_ortools(sizes, benefit, capacity, force_one, force_zero)
# solver_used = "BruteForce (fallback)"

# sol, obj = solve_pruned_ilp_bruteforce(sizes, benefit, capacity, force_one, force_zero)
# solver_used = "PuLP"


In [78]:
# Compute final latency given selection: sum latency_if_not_cached * (1 - L)
final_latency = np.sum(latency_if_not_cached * (1 - sol))

# Present results
df = pd.DataFrame({
    'item': np.arange(n_items),
    'size': sizes,
    'latency_if_not_cached_ms': np.round(latency_if_not_cached,2),
    'benefit': np.round(benefit,2),
    'recent_requests': recent_requests,
    'popularity': np.round(popularity,3),
    'q_value': np.round(q_values,4),
    'mask_phi': mask,
    'forced_one': force_one,
    'forced_zero': force_zero,
    'chosen_L': sol
})

# Try to use optional helper if available; otherwise display and save CSV
try:
    import caas_jupyter_tools as cjt
    cjt.display_dataframe_to_user("pruned_ilp_results", df)
    print("Displayed via caas_jupyter_tools.")
except Exception:
    try:
        from IPython.display import display
        display(df)
        print("Displayed via IPython.display. Also saving CSV...")
    except Exception:
        print("Falling back to CSV only...")
    df.to_csv("pruned_ilp_results.csv", index=False)
    print("Saved results to pruned_ilp_results.csv")

print(f"Solver used: {solver_used}")
print(f"Capacity: {capacity}, total sizes chosen: {int(np.dot(sol,sizes))}")
print(f"Objective (sum benefit of cached items): {obj:.2f}")
print(f"Final total latency (ms) after caching selection: {final_latency:.2f}")


,item,size,latency_if_not_cached_ms,benefit,recent_requests,popularity,q_value,mask_phi,forced_one,forced_zero,chosen_L
0,0,5,246.91,246.91,13,0.870,-0.0607,1,1,0,1
1,1,1,451.30,451.30,8,0.979,-0.3044,0,0,1,0
2,2,4,483.65,483.65,9,0.799,-0.0917,0,0,1,0
3,3,4,222.55,222.55,19,0.461,-0.0990,0,0,1,0
4,4,4,406.28,406.28,16,0.781,-0.0291,1,1,0,1
5,5,2,288.00,288.00,19,0.118,-0.1664,0,0,1,0
6,6,4,305.62,305.62,5,0.640,-0.0779,0,0,1,0
7,7,3,466.52,466.52,15,0.143,-0.0594,1,1,0,1


Displayed via IPython.display. Also saving CSV...
Saved results to pruned_ilp_results.csv
Solver used: BruteForce (fallback)
Capacity: 15, total sizes chosen: 12
Objective (sum benefit of cached items): 1119.71
Final total latency (ms) after caching selection: 1751.12
